<div style="width: 76ch;">
<h3>Ruitewissers</h3>
(<em>Colin Cools, Lander Cortens</em>)



</div>

In [ ]:
# Animation of four bar


import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib widget

In [ ]:
# kinematic parameters, in SI units:
r1   = 0.14 #m
r2   = 0.42 #m
r3   = 0.75 #m
r4   = 0.50 #m
r5   = 0.58 #m
r6   = 0.58 #m
r7   = 0.20 #m

rDC  = 0.17 #m
rDF  = 0.20 #m
rED  = 0.14 #m
rIJ  = 0.14 #m

grondA_x = 0.00 #m
grondA_y = 0.00 #m

grondD_x = 0.33 #m
grondD_y = 0.30 #m

grondH_x = 0.83 #m
grondH_y = 0.00 #m

grondI_x = 0.83 #m
grondI_y = 0.30 #m


rAD_x= abs(grondD_x - grondA_x)
rAD_y= abs(grondD_y - grondA_y)

rAH_x= abs(grondH_x - grondA_x)
rAH_y= abs(grondH_y - grondA_y)

rAI_x= abs(grondI_x - grondA_x)
rAI_y= abs(grondI_y - grondA_y)

rDH_x= abs(grondH_x - grondD_x)
rDH_y= abs(grondH_y - grondD_y)

rDI_x = abs(grondD_x - grondI_x)
rDI_y = abs(grondD_y - grondI_y)

assert np.isclose(rDI_x, r4), "Hard constraint violated: rDI_x must equal r4."
assert np.isclose(rED, rIJ), "Hard constraint violated: rED must equal rIJ."

closure_tol = 1e-8

omega  = 0.5      # driver frequency [rad/s]
t_begin =  0     # start time of simulation
t_end = 2*np.pi/omega     # end time of simulation
Ts      =  0.10  # target time step of simulation
num_steps = int(np.ceil((t_end - t_begin) / Ts))
t = np.linspace(t_begin, t_end, num_steps + 1)  # time vector for one full turn

# angular position driver:

theta1   = omega * t
dtheta1  = omega * np.ones_like(t)
ddtheta1 = np.zeros_like(t)

# configuration of numerical simulation interval and sampling:



# initial conditions for nonlinear solver ("fsolve") of position closure:
# phi1_init = 1.22 #rad volgens mij is deze bepaald door de motor, dus mag deze weg
theta2_init = 10*np.pi/180  #rad
theta3_init = 120*np.pi/180 #rad
theta4_init = 330*np.pi/180 #rad
theta5_init = 315*np.pi/180 #rad
theta6_init = 0*np.pi/180   #rad
theta7_init = 120*np.pi/180 #rad
# (different choices can lead to different topologies of your mechanism)

In [ ]:
# Function to rotate a vector z over an angle theta:
def rotate_vector(z, theta):
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta), np.cos(theta)]])
    return np.dot(rotation_matrix, z)

In [ ]:
# Define function to compute "gap" in position closure:
def loop_closure_eqs(theta_init, theta1, r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y):
    theta2 = theta_init[0]
    theta3 = theta_init[1]
    theta4 = theta_init[2]
    theta5 = theta_init[3]
    theta6 = theta_init[4]
    theta7 = theta_init[5]
    

    # Loop closure gaps:
    F1 = r1 * np.cos(theta1) + r2 * np.cos(theta2) + rDC * np.cos(theta3) - rAD_x
    F2 = r1 * np.sin(theta1) + r2 * np.sin(theta2) + rDC * np.sin(theta3) - rAD_y

    F3 = -rDF * np.cos(theta3) + r6 * np.cos(theta4) - r7 * np.cos(theta5) - rDH_x
    F4 = -rDF * np.sin(theta3) + r6 * np.sin(theta4) - r7 * np.sin(theta5) + rDH_y

    F5 = rED * np.cos(theta3) + r4 * np.cos(theta6) - rIJ * np.cos(theta7) - rDI_x
    F6 = rED * np.sin(theta3) + r4 * np.sin(theta6) - rIJ * np.sin(theta7) - rDI_y

    return [F1, F2, F3, F4, F5, F6]

In [ ]:
# calculating unknown angles at time zero
def angles_at_time_zero(r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y, theta1, dtheta1, ddtheta1, theta2_init, theta3_init,theta4_init,theta5_init,theta6_init,theta7_init, t):
    optim_options = {"full_output":True}  # options for fsolve

  # numerically solving the position closure at all sampling times:
    #for k, time in enumerate(t):
    #for k, time in [0]:
    k=0
    if k ==0:
      # Position Analysis
      x, _, ier, message  = fsolve(lambda x: loop_closure_eqs(x, theta1[k], r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y), [theta2_init, theta3_init,theta4_init,theta5_init,theta6_init,theta7_init], **optim_options)

      residual = np.linalg.norm(loop_closure_eqs(x, theta1[k], r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y))

      if ier != 1 or residual > closure_tol:
          raise ValueError(
              f"Initial position solve failed at k={k}, t={t[k]:.3f} s, theta1={theta1[k]:.3f} rad, residual={residual:.3e}. {message}"
          )

      theta2 = x[0]
      theta3 = x[1]
      theta4 = x[2]
      theta5 = x[3]
      theta6 = x[4]
      theta7 = x[5]

      

    return theta2, theta3, theta4, theta5, theta6, theta7

theta2_0, theta3_0, theta4_0, theta5_0, theta6_0, theta7_0 = angles_at_time_zero(r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y, theta1, dtheta1, ddtheta1, theta2_init, theta3_init,theta4_init,theta5_init,theta6_init,theta7_init, t)
print("theta1:", theta1[0])
print("theta2:", theta2_0)
print("theta3:", theta3_0)
print("theta4:", theta4_0)
print("theta5:", theta5_0)
print("theta6:", theta6_0)
print("theta7:", theta7_0)

In [ ]:
# making plot at time zero
import numpy as np
import matplotlib.pyplot as plt


A = np.array([0, 0])
B = A+ rotate_vector(np.array([r1, 0]), theta1[0])
C = B + rotate_vector(np.array([r2, 0]), theta2_0)
D = A + np.array([rAD_x, rAD_y])
I = A + np.array([rAI_x, rAI_y])
H = A + np.array([rAH_x, rAH_y])
G = H + rotate_vector(np.array([r7, 0]), theta5_0)
F = G - rotate_vector(np.array([r6, 0]), theta4_0)
K = F + rotate_vector(np.array([r3, 0]), theta3_0)
E = F + rotate_vector(np.array([rED + rDF, 0]), theta3_0)
J = I + rotate_vector(np.array([rIJ, 0]), theta7_0)
L = I + rotate_vector(np.array([r5, 0]), theta7_0)



plt.figure()
pad1 = np.array([H, G, F, D, E, J, I])
plt.plot(pad1[:, 0], pad1[:, 1], '-bo')
pad2 = np.array([A, B,C])
plt.plot(pad2[:, 0], pad2[:, 1], '-bo')
pad3 = np.array([E,K])
plt.plot(pad3[:, 0], pad3[:, 1], '-bo')
pad4 = np.array([J,L])
plt.plot(pad4[:, 0], pad4[:, 1], '-bo')

plt.plot(A[0], A[1], 'ro')  # vaste pivot
plt.plot(D[0], D[1], 'ro')
plt.plot(H[0], H[1], 'ro')
plt.plot(I[0], I[1], 'ro')
plt.xlabel('[m]')
plt.ylabel('[m]')
plt.title('Assembly')
plt.axis('equal')
plt.show()




In [ ]:
# Function to compute the kinematics
def kinematics(r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y, theta1, dtheta1, ddtheta1, theta2_init, theta3_init,theta4_init,theta5_init,theta6_init,theta7_init, t):
    optim_options = {"full_output":True}  # options for fsolve

  # numerically solving the position closure at all sampling times:
    for k, time in enumerate(t):
      # Position Analysis
      x, _, ier, message  = fsolve(lambda x: loop_closure_eqs(x, theta1[k], r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y), [theta2_init, theta3_init,theta4_init,theta5_init,theta6_init,theta7_init], **optim_options)

      residual = np.linalg.norm(loop_closure_eqs(x, theta1[k], r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y))

      if ier != 1 or residual > closure_tol:
          raise ValueError(
              f"Position solve failed at k={k}, t={time:.3f} s, theta1={theta1[k]:.3f} rad, residual={residual:.3e}. {message}"
          )

      theta2[k] = x[0]
      theta3[k] = x[1]
      theta4[k] = x[2]
      theta5[k] = x[3]
      theta6[k] = x[4]
      theta7[k] = x[5]

      # velocity closure:
      A = np.array([
        [-r2*np.sin(theta2[k]), -rDC*np.sin(theta3[k]), 0, 0, 0, 0],
        [ r2*np.cos(theta2[k]),  rDC*np.cos(theta3[k]), 0, 0, 0, 0],
        [0,  rDF*np.sin(theta3[k]), -r6*np.sin(theta4[k]),  r7*np.sin(theta5[k]), 0, 0],
        [0, -rDF*np.cos(theta3[k]),  r6*np.cos(theta4[k]), -r7*np.cos(theta5[k]), 0, 0],
        [0, -rED*np.sin(theta3[k]), 0, 0, -r4*np.sin(theta6[k]),  rIJ*np.sin(theta7[k])],
        [0,  rED*np.cos(theta3[k]), 0, 0,  r4*np.cos(theta6[k]), -rIJ*np.cos(theta7[k])]
      ])

      B = np.array([
        r1*np.sin(theta1[k])*dtheta1[k],
        -r1*np.cos(theta1[k])*dtheta1[k],
        0,
        0,
        0,
        0
      ])

      x = np.linalg.solve(A, B)
      cond[k]  = np.linalg.cond(A)

      dtheta2[k] = x[0]
      dtheta3[k] = x[1]
      dtheta4[k] = x[2]
      dtheta5[k] = x[3]
      dtheta6[k] = x[4]
      dtheta7[k] = x[5]

      A = np.array([
        [-r2*np.sin(theta2[k]), -rDC*np.sin(theta3[k]), 0, 0, 0, 0],
        [ r2*np.cos(theta2[k]),  rDC*np.cos(theta3[k]), 0, 0, 0, 0],
        [0,  rDF*np.sin(theta3[k]), -r6*np.sin(theta4[k]),  r7*np.sin(theta5[k]), 0, 0],
        [0, -rDF*np.cos(theta3[k]),  r6*np.cos(theta4[k]), -r7*np.cos(theta5[k]), 0, 0],
        [0, -rED*np.sin(theta3[k]), 0, 0, -r4*np.sin(theta6[k]),  rIJ*np.sin(theta7[k])],
        [0,  rED*np.cos(theta3[k]), 0, 0,  r4*np.cos(theta6[k]), -rIJ*np.cos(theta7[k])]
      ])

      B = np.array([
        r1*np.cos(theta1[k])*dtheta1[k]**2 + r1*np.sin(theta1[k])*ddtheta1[k]
        + r2*np.cos(theta2[k])*dtheta2[k]**2 + rDC*np.cos(theta3[k])*dtheta3[k]**2,

        r1*np.sin(theta1[k])*dtheta1[k]**2 - r1*np.cos(theta1[k])*ddtheta1[k]
        + r2*np.sin(theta2[k])*dtheta2[k]**2 + rDC*np.sin(theta3[k])*dtheta3[k]**2,

        -rDF*np.cos(theta3[k])*dtheta3[k]**2 + r6*np.cos(theta4[k])*dtheta4[k]**2 - r7*np.cos(theta5[k])*dtheta5[k]**2,

        -rDF*np.sin(theta3[k])*dtheta3[k]**2 + r6*np.sin(theta4[k])*dtheta4[k]**2 - r7*np.sin(theta5[k])*dtheta5[k]**2,

        rED*np.cos(theta3[k])*dtheta3[k]**2 + r4*np.cos(theta6[k])*dtheta6[k]**2 - rIJ*np.cos(theta7[k])*dtheta7[k]**2,

        rED*np.sin(theta3[k])*dtheta3[k]**2 + r4*np.sin(theta6[k])*dtheta6[k]**2 - rIJ*np.sin(theta7[k])*dtheta7[k]**2
      ])

      x = np.linalg.solve(A, B)

      ddtheta2[k] = x[0]
      ddtheta3[k] = x[1]
      ddtheta4[k] = x[2]
      ddtheta5[k] = x[3]
      ddtheta6[k] = x[4]
      ddtheta7[k] = x[5] 

      theta2_init = theta2[k] + (t[1] - t[0])*dtheta2[k]
      theta3_init = theta3[k] + (t[1] - t[0])*dtheta3[k]
      theta4_init = theta4[k] + (t[1] - t[0])*dtheta4[k]
      theta5_init = theta5[k] + (t[1] - t[0])*dtheta5[k]
      theta6_init = theta6[k] + (t[1] - t[0])*dtheta6[k]
      theta7_init = theta7[k] + (t[1] - t[0])*dtheta7[k]

    return theta2, theta3, theta4, theta5, theta6, theta7, dtheta2, dtheta3, dtheta4, dtheta5, dtheta6, dtheta7, ddtheta2, ddtheta3, ddtheta4, ddtheta5, ddtheta6, ddtheta7

In [ ]:
theta2   = np.zeros_like(t)
theta3   = np.zeros_like(t)
theta4   = np.zeros_like(t)
theta5   = np.zeros_like(t)
theta6   = np.zeros_like(t)
theta7   = np.zeros_like(t)

dtheta2   = np.zeros_like(t)
dtheta3   = np.zeros_like(t)
dtheta4   = np.zeros_like(t)
dtheta5   = np.zeros_like(t)
dtheta6   = np.zeros_like(t)
dtheta7   = np.zeros_like(t)

ddtheta2   = np.zeros_like(t)
ddtheta3   = np.zeros_like(t)
ddtheta4   = np.zeros_like(t)
ddtheta5   = np.zeros_like(t)
ddtheta6   = np.zeros_like(t)
ddtheta7   = np.zeros_like(t)

cond   = np.zeros_like(t) # condition number of matrix A

t_size = len(t)              # number of iterations
sim_fraction = 2             # fraction of simulated frames in animation
frames = int(t_size / sim_fraction)# number of animated iterations
delta = int(np.floor(t_size/frames))    # time between frames
index_vec = np.arange(0, t_size, delta) # index of animated frames  

theta2, theta3, theta4, theta5, theta6, theta7, dtheta2, dtheta3, dtheta4, dtheta5, dtheta6, dtheta7, ddtheta2, ddtheta3, ddtheta4, ddtheta5, ddtheta6, ddtheta7 = kinematics(r1, r2, r3, r4, r5, r6, r7, rDC, rDF, rED, rIJ, rAD_x, rAD_y, rDI_x, rDI_y, theta1, dtheta1, ddtheta1, theta2_init, theta3_init,theta4_init,theta5_init,theta6_init,theta7_init, t)

In [ ]:
# Animation of mechanism motion
plt.ioff()               # don't show figure until drawn explicitly:
fig2, ax = plt.subplots() # Initialize the figure

# compute width and height of drawing canvas:
x_left   = -1.25 * max(r1, r3 -rAD_x - rDF)
y_bottom = -1.25 * max(r1, r7)
x_right  = rAH_x + 1.25 * max(r7, r5)
y_top    = rAD_y + 1.25 * max(r5, r3 -rDF)

# function to update the animation every "delta" steps:
def update(frame):
    global fig2, ax, x_left, x_right, y_bottom, y_top, P, S, index_vec
    ax.cla()         # Clear the current axes
    ax.axis('equal') # Ensures 1 unit on x == 1 unit on y
    ax.set_xlabel('[m]')
    ax.set_ylabel('[m]')
    ax.set_xlim([x_left, x_right])
    ax.set_ylim([y_bottom, y_top])
    time_now = t[index_vec[frame]]
    ax.set_title(f'Time {time_now: .2f}')
    #ax.set_title(f'Frame {frame}')

    # Calculate Cartesian coordinates of mechanism joints:
    A = np.array([0, 0])
    B = A+ rotate_vector(np.array([r1, 0]), theta1[index_vec[frame]])
    C = B + rotate_vector(np.array([r2, 0]), theta2[index_vec[frame]])
    D = A + np.array([rAD_x, rAD_y])
    I = A + np.array([rAI_x, rAI_y])
    H = A + np.array([rAH_x, rAH_y])
    G = H + rotate_vector(np.array([r7, 0]), theta5[index_vec[frame]])
    F = G - rotate_vector(np.array([r6, 0]), theta4[index_vec[frame]])
    K = F + rotate_vector(np.array([r3, 0]), theta3[index_vec[frame]])
    E = F + rotate_vector(np.array([rED + rDF, 0]), theta3[index_vec[frame]])
    J = I + rotate_vector(np.array([rIJ, 0]), theta7[index_vec[frame]])
    L = I + rotate_vector(np.array([r5, 0]), theta7[index_vec[frame]])

    pad1 = np.array([H, G, F, C, D, E, J, I])
    ax.plot(pad1[:, 0], pad1[:, 1], '-bo')
    pad2 = np.array([A, B,C])
    ax.plot(pad2[:, 0], pad2[:, 1], '-bo')
    pad3 = np.array([E,K])
    ax.plot(pad3[:, 0], pad3[:, 1], '-bo')
    pad4 = np.array([J,L])
    ax.plot(pad4[:, 0], pad4[:, 1], '-bo')

    ax.plot(A[0], A[1], 'ro')  # vaste pivot
    ax.plot(D[0], D[1], 'ro')
    ax.plot(H[0], H[1], 'ro')
    ax.plot(I[0], I[1], 'ro')
    return(ax)

# animate the mechanism:
ani = FuncAnimation(fig2, update, frames=len(index_vec), interval=50, repeat=False)
# display mechanism animation:
ani_html = ani.to_jshtml(default_mode='once')
HTML(ani_html)